# Mass-Spring-Damper Forecasting — Physics-Based Approach

This notebook forecasts the **full evaluation window** of a mass-spring-damper
system using a physics-based approach: estimate system parameters from the
observation phase, then integrate the equations of motion forward for every
step in the evaluation window.

## Two-phase contest structure

| Phase | What happens |
|-------|--------------|
| **Observation** | The simulation runs and you receive live sensor readings. Collect data and fit the model here. |
| **Evaluation** | Simulation continues but the stream is closed. Ground truth is recorded. |
| **Submission** | Submit your forecast — one list of `eval_steps` predicted values per sensor. |

## System dynamics

The system obeys:

$$m \ddot{x} + c \dot{x} + k x = F(t)$$

where:
- $m = 1.0$ kg — mass (known)
- $c$ — damping coefficient (unknown, estimated from data)
- $k$ — spring stiffness (unknown, possibly degraded by a fault)
- $F(t) = 0.5 \sin(2\pi t)$ — external harmonic forcing (known)

## Strategy

1. Collect a window of live position observations during the observation phase.
2. Estimate $k$ and $c$ from the data using least squares.
3. Integrate the equation of motion forward for `eval_steps` steps to generate a full-sequence forecast.
4. Submit the forecast once the submission window opens.

## Installation — run this cell once

In [ ]:
%pip install "epic-elios-client[notebook]" matplotlib numpy

## Configure the Client

In [ ]:
from epic_client import EPICClient

SERVER_URL = "https://epic.elioslab.net"

# Mass is known from the system specification.
MASS = 1.0

client = EPICClient(SERVER_URL)

## 1. Log In

In [ ]:
client.login("your-username", "your-password")

## 2. Browse Available Contests

Look for a contest based on the `mass_spring_damper` twin with status `ACTIVE`.

In [ ]:
import pandas as pd

contests = client.list_contests(status="ACTIVE")

rows = []
for c in contests:
    task_cfg = c.get("tasks", [{}])[0].get("configuration", {})
    rows.append({
        "contest_id":                c.get("contest_id"),
        "name":                      c.get("name"),
        "twin_id":                   c.get("twin_id"),
        "sampling_rate_hz":          c.get("sampling_rate_hz"),
        "end_of_observation":        c.get("end_of_observation"),
        "prediction_horizon_seconds": c.get("prediction_horizon_seconds"),
        "eval_steps":                task_cfg.get("eval_steps"),
    })

pd.DataFrame(rows)

Copy a `contest_id` from the table above.

In [ ]:
CONTEST_ID = "replace-with-contest-id"

# Read contest parameters.
contest = next(c for c in contests if c["contest_id"] == CONTEST_ID)
SAMPLING_RATE_HZ = contest["sampling_rate_hz"]
DT = 1.0 / SAMPLING_RATE_HZ
EVAL_STEPS = contest["tasks"][0]["configuration"]["eval_steps"]

print(f"sampling_rate_hz : {SAMPLING_RATE_HZ} Hz  (dt = {DT:.4f} s)")
print(f"eval_steps       : {EVAL_STEPS}")
print(f"evaluation window: {EVAL_STEPS * DT:.1f} s  ({EVAL_STEPS} steps × {DT:.4f} s)")

## 3. Register for the Contest

In [ ]:
client.register(CONTEST_ID)

## 4. Collect Live Observations — Observation Phase

Collect a window long enough to estimate system parameters reliably.
At least 30 seconds is recommended — more data gives a better fit.
The stream closes automatically when the observation phase ends.

> **Tip:** set `duration_seconds` to the full length of the observation window
> so you get as much data as possible.

In [ ]:
observations = await client.collect(
    CONTEST_ID,
    duration_seconds=60,          # adjust to match the observation window
    csv_path="msd_observations.csv",
)
print(f"Collected {len(observations)} observations  ({len(observations) * DT:.1f} s)")

## 5. Build the Dataset

In [ ]:
import numpy as np

rows = []
for obs in observations:
    rows.append({
        "sequence_id": obs["sequence_id"],
        "timestamp":   obs["timestamp"],
        **obs["sensors"],
    })

df = pd.DataFrame(rows)
df["t"] = (df["sequence_id"] - df["sequence_id"].iloc[0]) * DT
df.head()

## 6. Plot the Position Signal

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.plot(df["t"], df["position"], linewidth=1.2, label="position")
plt.xlabel("Time (s)")
plt.ylabel("Position (m)")
plt.title("Live position stream — Mass-Spring-Damper")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Estimate System Parameters

We discretise the equation of motion:

$$m a_i + c v_i + k x_i = F(t_i)$$

and rearrange:

$$k x_i + c v_i = F(t_i) - m a_i$$

Velocity and acceleration are approximated by central finite differences:

$$v_i = \frac{x_{i+1} - x_{i-1}}{2 \Delta t}, \quad a_i = \frac{x_{i+1} - 2x_i + x_{i-1}}{\Delta t^2}$$

The system $[x_i,\, v_i]\,[k,\,c]^T = b_i$ is solved in the least-squares sense.

In [ ]:
x = df["position"].values
t = df["t"].values

# Central finite differences (exclude first and last points)
v = (x[2:] - x[:-2]) / (2 * DT)
a = (x[2:] - 2 * x[1:-1] + x[:-2]) / (DT ** 2)
x_mid = x[1:-1]
t_mid = t[1:-1]

# External forcing F(t) = 0.5 * sin(2π * t)
F = 0.5 * np.sin(2 * np.pi * t_mid)

# Right-hand side: F(t) - m*a
b = F - MASS * a

# Design matrix [x, v]
A = np.column_stack([x_mid, v])

# Least squares: solve for [k, c]
params, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
k_est, c_est = params

print(f"Estimated stiffness  k = {k_est:.4f}  (nominal: 10.0)")
print(f"Estimated damping    c = {c_est:.4f}  (nominal:  0.5)")

## 8. Validate the Fit

Reconstruct the acceleration from the estimated parameters and compare it
against the finite-difference acceleration.

In [ ]:
a_pred = (F - k_est * x_mid - c_est * v) / MASS

plt.figure(figsize=(12, 4))
plt.plot(t_mid, a, label="measured (finite diff)", linewidth=1.2)
plt.plot(t_mid, a_pred, label="model fit", linewidth=1.2, linestyle="--")
plt.xlabel("Time (s)")
plt.ylabel("Acceleration (m/s²)")
plt.title("Parameter fit validation")
plt.legend()
plt.tight_layout()
plt.show()

rmse = np.sqrt(np.mean((a - a_pred) ** 2))
print(f"Fit RMSE (acceleration): {rmse:.6f} m/s²")

## 9. Forecast the Full Evaluation Window

Integrate the equation of motion forward from the last observed state for
exactly **`eval_steps` steps** using the 4th-order Runge-Kutta method.

The state vector is $[x, v]$ and the dynamics are:

$$\dot{x} = v, \quad \dot{v} = \frac{F(t) - c\,v - k\,x}{m}$$

In [ ]:
def msd_rhs(t, x, v, k, c, m):
    F = 0.5 * np.sin(2 * np.pi * t)
    return v, (F - c * v - k * x) / m


def rk4_step(t, x, v, dt, k, c, m):
    k1x, k1v = msd_rhs(t,        x,              v,              k, c, m)
    k2x, k2v = msd_rhs(t + dt/2, x + dt/2*k1x,  v + dt/2*k1v,  k, c, m)
    k3x, k3v = msd_rhs(t + dt/2, x + dt/2*k2x,  v + dt/2*k2v,  k, c, m)
    k4x, k4v = msd_rhs(t + dt,   x + dt*k3x,    v + dt*k3v,    k, c, m)
    x_new = x + (dt / 6) * (k1x + 2*k2x + 2*k3x + k4x)
    v_new = v + (dt / 6) * (k1v + 2*k2v + 2*k3v + k4v)
    return x_new, v_new


# Initial state from the last two observations.
x0 = x[-1]
v0 = (x[-1] - x[-2]) / DT
t0 = t[-1]

# Integrate forward for eval_steps steps — one predicted value per step.
x_curr, v_curr, t_curr = x0, v0, t0
position_forecast = []

for _ in range(EVAL_STEPS):
    x_curr, v_curr = rk4_step(t_curr, x_curr, v_curr, DT, k_est, c_est, MASS)
    t_curr += DT
    position_forecast.append(x_curr)

print(f"Forecast length : {len(position_forecast)} steps  (expected {EVAL_STEPS})")
print(f"First 5 values  : {[round(v, 4) for v in position_forecast[:5]]}")
print(f"Last  5 values  : {[round(v, 4) for v in position_forecast[-5:]]}")

## 10. Plot the Forecast

Visualise the full forecast sequence against the observation history.

In [ ]:
t_forecast = t[-1] + DT * (np.arange(1, EVAL_STEPS + 1))

plt.figure(figsize=(12, 4))
plt.plot(t, x, label="observed", linewidth=1.2)
plt.plot(t_forecast, position_forecast, label=f"forecast ({EVAL_STEPS} steps)",
         linewidth=1.5, linestyle="--", color="tab:orange")
plt.axvline(t[-1], color="gray", linestyle=":", label="observation end")
plt.xlabel("Time (s)")
plt.ylabel("Position (m)")
plt.title("Mass-Spring-Damper — Observation History + Forecast")
plt.legend()
plt.tight_layout()
plt.show()

## 11. Build and Submit the Prediction

The payload must contain exactly `eval_steps` values for every sensor.
Submissions are accepted only **after the evaluation window closes**.

In [ ]:
payload = {
    "forecast": {
        "position": [float(v) for v in position_forecast],
    }
}

print(f"Payload: {len(payload['forecast']['position'])} predicted values")

submission = client.submit(
    contest_id=CONTEST_ID,
    task_id="forecasting",
    payload=payload,
)
print("Submission:", submission)

## 12. Check Scores and Leaderboard

In [ ]:
scores = client.get_scores(CONTEST_ID)
leaderboard = client.get_leaderboard(CONTEST_ID)

print("=== My Submissions ===")
for sub in scores.get("submissions", []):
    print(f"  {sub['submission_id'][:8]}…  status={sub['status']}  "
          f"scores={[round(s['value'], 4) for s in sub.get('scores', [])]}")

print("\n=== Leaderboard ===")
for entry in leaderboard.get("entries", []):
    print(f"  Rank {entry['rank']}  user={entry['user_id'][:8]}…  score={entry['score']:.4f}")

## Next Steps

This approach estimates constant $k$ and $c$ over the full observation window.
Under the `reduced_stiffness` fault, $k$ drifts over time — the estimate will
be an average, which degrades accuracy at the end of long forecasts.

Possible improvements:

- **Sliding-window estimation** — re-estimate $k$ and $c$ using only the most
  recent observations to track parameter drift.
- **Recursive least squares** — update parameter estimates online without
  refitting from scratch.
- **Kalman filter** — jointly estimate state and parameters in a statistically
  optimal way; handles measurement noise gracefully.
- **Data-driven models** — train an LSTM or Transformer on historical data
  collected at various fault severities.